In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, log_loss
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from capstone.config import CATEGORICAL, FEATURES, NUMERIC, STYLES, CVConfig
from capstone.data import load_recipes
from capstone.features import build_preprocessor
from capstone.models import AffinityClassifier
from capstone.utilities import setup_logging

setup_logging("WARNING")

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
df = load_recipes()
X, y = df[FEATURES], df["style"]
CV = CVConfig()
N_SPLITS, N_REPEATS, SEED = CV.n_splits, CV.n_repeats, CV.seed
cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
y.value_counts()

In [ ]:
def make_models(num=NUMERIC, cat=CATEGORICAL):
    estimators = {
        "chance (stratified)": DummyClassifier(strategy="stratified", random_state=SEED),
        "affinity baseline": AffinityClassifier(),
        "logistic regression": LogisticRegression(C=1.0, max_iter=1000),
        "k-NN (k=3)": KNeighborsClassifier(n_neighbors=3),
        "k-NN (k=5)": KNeighborsClassifier(n_neighbors=5),
        "decision tree": DecisionTreeClassifier(max_depth=3, random_state=SEED),
        "random forest": RandomForestClassifier(n_estimators=300, random_state=SEED),
        "SVM (rbf)": SVC(probability=True, random_state=SEED),
    }
    return {name: Pipeline([("prep", build_preprocessor(num, cat)), ("clf", est)]) for name, est in estimators.items()}

In [ ]:
def run_cv(models, X, y, cv):
    """Per-fold metrics and out-of-fold predictions for every model, same splits for all."""
    fold_rows, pred_rows = [], []
    for fold_id, (tr, te) in enumerate(cv.split(X, y)):
        repeat, fold = divmod(fold_id, cv.cvargs["n_splits"])
        for name, model in models.items():
            m = clone(model).fit(X.iloc[tr], y.iloc[tr])
            pred = m.predict(X.iloc[te])
            proba = m.predict_proba(X.iloc[te])
            fold_rows.append({
                "model": name, "repeat": repeat, "fold": fold,
                "accuracy": accuracy_score(y.iloc[te], pred),
                "macro_f1": f1_score(y.iloc[te], pred, average="macro", labels=STYLES),
                "log_loss": log_loss(y.iloc[te], proba, labels=m.classes_),
            })
            pred_rows.append(pd.DataFrame({"model": name, "repeat": repeat, "recipe_id": X.index[te],
                                           "true": y.iloc[te].values, "pred": pred}))
    return pd.DataFrame(fold_rows), pd.concat(pred_rows, ignore_index=True)

folds, preds = run_cv(make_models(), X, y, cv)
folds.head()

In [ ]:
# Distribution of each metric across all folds (5 folds x 20 repeats)
summary = folds.groupby("model")[["accuracy", "macro_f1", "log_loss"]].agg(["mean", "std"])
summary.sort_values(("accuracy", "mean"), ascending=False)

In [ ]:
order = folds.groupby("model")["accuracy"].mean().sort_values(ascending=False).index
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, metric in zip(axes, ["accuracy", "macro_f1"]):
    ax.boxplot([folds.loc[folds["model"] == m, metric] for m in order], tick_labels=order)
    ax.axhline(folds.loc[folds["model"] == "affinity baseline", metric].mean(), color="tab:red", ls="--", label="affinity baseline mean")
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=40)
    ax.legend()
fig.tight_layout()

In [ ]:
# Repeat-level view (each point = mean over the 5 folds of one repeat), which is the unit that varies independently of fold size
by_repeat = folds.groupby(["model", "repeat"])[["accuracy", "macro_f1"]].mean().reset_index()
by_repeat.groupby("model")["accuracy"].describe()[["mean", "std", "min", "25%", "50%", "75%", "max"]].loc[order]

In [ ]:
# Paired comparison against the affinity baseline on identical folds
base = folds[folds["model"] == "affinity baseline"].set_index(["repeat", "fold"])
rows = []
for name in order:
    if name == "affinity baseline":
        continue
    other = folds[folds["model"] == name].set_index(["repeat", "fold"])
    diff = (other["accuracy"] - base["accuracy"])
    per_repeat = diff.groupby("repeat").mean()
    rows.append({
        "model": name,
        "mean_acc_diff": diff.mean(),
        "repeat_5pct": per_repeat.quantile(0.05),
        "repeat_95pct": per_repeat.quantile(0.95),
        "folds_better": (diff > 0).mean(),
        "folds_tied": (diff == 0).mean(),
        "folds_worse": (diff < 0).mean(),
        "f1_diff": (other["macro_f1"] - base["macro_f1"]).mean(),
        "logloss_diff": (other["log_loss"] - base["log_loss"]).mean(),
    })
pd.DataFrame(rows).set_index("model")

In [ ]:
# Out-of-fold confusion matrices (counts summed across repeats, normalized by true-class totals)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, name in zip(axes.ravel(), order):
    p = preds[preds["model"] == name]
    cm = confusion_matrix(p["true"], p["pred"], labels=STYLES).astype(float)
    cm /= cm.sum(axis=1, keepdims=True)
    ax.imshow(cm, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(3), STYLES, rotation=30)
    ax.set_yticks(range(3), STYLES)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center")
    ax.set_title(name)
fig.tight_layout()

In [ ]:
# Per-class recall and precision (out-of-fold, pooled over repeats)
rows = []
for name in order:
    p = preds[preds["model"] == name]
    f1s = f1_score(p["true"], p["pred"], labels=STYLES, average=None)
    for s, f in zip(STYLES, f1s):
        rows.append({"model": name, "style": s, "f1": f})
pd.DataFrame(rows).pivot(index="model", columns="style", values="f1").loc[order]

In [ ]:
# Per-recipe: how often each model gets the expert label right across repeats
preds["correct"] = preds["true"] == preds["pred"]
per_recipe = preds.pivot_table(index="recipe_id", columns="model", values="correct", aggfunc="mean")[list(order)]
per_recipe.insert(0, "style", y)
per_recipe["mean_all_models"] = per_recipe[[m for m in order if not m.startswith("chance")]].mean(axis=1)
per_recipe.sort_values("mean_all_models").head(12)

In [ ]:
# Recipes the baseline gets wrong vs. recipes the models fix or break
fixed = per_recipe[(per_recipe["affinity baseline"] < 0.5) & (per_recipe[["logistic regression", "random forest", "k-NN (k=3)", "SVM (rbf)"]].mean(axis=1) >= 0.5)]
broken = per_recipe[(per_recipe["affinity baseline"] >= 0.5) & (per_recipe[["logistic regression", "random forest", "k-NN (k=3)", "SVM (rbf)"]].mean(axis=1) < 0.5)]
print("baseline wrong, other models mostly right:", list(fixed.index))
print("baseline right, other models mostly wrong:", list(broken.index))

In [ ]:
# Most common predicted label for hard recipes (what the models think they are)
hard = per_recipe[per_recipe["mean_all_models"] < 0.5].index
(preds[preds["recipe_id"].isin(hard) & ~preds["model"].str.startswith("chance")]
 .groupby(["recipe_id", "true"])["pred"].value_counts(normalize=True).unstack(fill_value=0))

In [ ]:
# Which feature groups carry the signal: rerun with numeric-only and categorical-only
subsets = {"all": (NUMERIC, CATEGORICAL), "numeric only": (NUMERIC, []), "categorical only": ([], CATEGORICAL)}
keep = ["affinity baseline", "logistic regression", "k-NN (k=3)", "random forest", "SVM (rbf)"]
ablation = {}
for label, (num, cat) in subsets.items():
    models = {k: v for k, v in make_models(num, cat).items() if k in keep}
    f, _ = run_cv(models, X, y, cv)
    ablation[label] = f.groupby("model")[["accuracy", "macro_f1"]].mean()
pd.concat(ablation, axis=1).loc[keep]

In [ ]:
# Per-feature ablation: drop one feature at a time, measure accuracy change vs. all features
drop_rows = []
for feat in FEATURES:
    num = [f for f in NUMERIC if f != feat]
    cat = [f for f in CATEGORICAL if f != feat]
    models = {k: v for k, v in make_models(num, cat).items() if k in keep}
    f, _ = run_cv(models, X, y, cv)
    acc = f.groupby("model")["accuracy"].mean()
    for m in keep:
        drop_rows.append({"dropped": feat, "model": m, "accuracy": acc[m]})
drop = pd.DataFrame(drop_rows).pivot(index="dropped", columns="model", values="accuracy")[keep]
drop.sub(folds[folds["model"].isin(keep)].groupby("model")["accuracy"].mean(), axis=1).loc[FEATURES]

In [ ]:
# Interpretation: logistic-regression coefficients and random-forest permutation importance, fit on all data
lr = make_models()["logistic regression"].fit(X, y)
names = lr["prep"].get_feature_names_out()
coef = pd.DataFrame(lr["clf"].coef_, index=lr["clf"].classes_, columns=names).T
coef.reindex(coef.abs().max(axis=1).sort_values(ascending=False).index).head(12)

In [ ]:
rf = make_models()["random forest"]
perm_rows = []
for tr, te in RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=5, random_state=SEED).split(X, y):
    m = clone(rf).fit(X.iloc[tr], y.iloc[tr])
    r = permutation_importance(m, X.iloc[te], y.iloc[te], n_repeats=10, random_state=SEED, scoring="accuracy")
    perm_rows.append(pd.Series(r.importances_mean, index=FEATURES))
perm = pd.concat(perm_rows, axis=1)
perm.mean(axis=1).sort_values().plot.barh(xerr=perm.std(axis=1).reindex(perm.mean(axis=1).sort_values().index), figsize=(6, 4), title="RF permutation importance (held-out)")
plt.tight_layout()

In [ ]:
# Does model-agreement track the labels? Fraction of recipes where all non-chance models agree, and agree with the label
valid = preds[~preds["model"].str.startswith("chance")]
agree = valid.groupby(["repeat", "recipe_id"]).agg(n_unique=("pred", "nunique"), all_correct=("correct", "all"))
print("all models predict the same style:", (agree["n_unique"] == 1).mean().round(3))
print("all models match the expert label:", agree["all_correct"].mean().round(3))